#### Merges zotero entry pdf or html attachments to groups of markdown files
with metadata separating them that's supposed to be readable by NotebookLM and 

In [3]:
%load_ext autoreload
%autoreload 2

from icecream import ic
import pathlib as pl
from pyzotero import zotero
import sys
import pandas as pd
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import mdformat
import datetime as dt
from pathlib import Path
import re

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw

In [2]:
output_md_file_min_bytes = 4000  # file declared bad if < this
maxNwordsMergedGroup = 450000  # Notebook LM official limit is 500K
save_merged_pdfs = False # if true save pdfs of grouped and merged markdown files

In [ ]:
# Collections to get and convert

# I got these using this command:
#
# collections_in_Politics = [c for c in rfw.get_my_zotero_collections('Politics') 
#                         if c not in rfw.get_my_zotero_collections('CivicSatCarolineK2024')]
# # don't get top level, doing this would miss articles stored immediately below it (none there, last time I checked)
# collections_in_Politics = list(set(collections_to_Politics) - set('Politics')) 

# collections that will be merged into a single markdown file
print("This grouping scheme will miss any entry immediately under 'Politics' (none, last time I checked)")
collection_groups_to_get = dict(
    Prediction=['PoliticalML', 'PollMethods', 'FocusGroups', 'ElectionPredFeats'], 
    Messaging=['MisDisinformation','MediaAdsPolit'], 
    Cognitive=['NeuroPsychoLinguisticPolitics', 'Polarization', 'IdentityPolitics'], 
    HotTakes=['Hot Takes US Elect 2024', 'SuccessStores'], 
    Organization=['PartyOrganization', 'CampaignMoney', 'Voting Systems'])

# All the collections that have to be converted to markdown
collections_to_markdown = list({collection for group in collection_groups_to_get.values() for collection in group})
#collections_to_markdown

This grouping scheme will miss any entry immediately under 'Politics' (none, last time I checked)


In [4]:
zotinfo = rfw.load_pickle_data(rfw.extractedZoteroEntriesFNm) # all attachments from load_zotero_data.ipynb

# select which attachments to merge
attachments = zotinfo['attachment_files'] # .query('contentType == "text/html"')
doGetAttachment = attachments['parentCollections'].apply(lambda cThis:  any(c in cThis for c in collections_to_markdown))

attachments = attachments[doGetAttachment]

# Standardize merged output separator page meta information names
metainfo_renames = dict(Title='parentTitle',Author='parentFirstCreator',Source='parentVenue', Date='parentDate')
metaColNms = metainfo_renames.values()
#metaCols = attachments[metainfo_renames.values()].rename(metainfo_renames, axis=1)

attachments = attachments.rename(metainfo_renames, axis=1)

Reading from C:\Users\scott\OneDrive\share\ref\refwrangle\dat\zotero_entries.pkl...


In [5]:
# Functions for converting attachments to md: preferred source type is 1st key in this dict declaration
convert_source_to_markdown_func = {'application/pdf':rfw.pdf2md_pymupdf4llm, 'text/html':rfw.html2md_cautious}

def is_markdown_file_big_and_current(fileInfo, failure_log=None):
    """Return True if fileInfo['markdown_file_path'] is big enough for a markdown file 
    and is up to date, relative to the source file it was made from, fileInfo['source']"""

    isBig = is_markdown_file_big_enough(fileInfo, failure_log)
    if isBig:
        if (fileInfo['markdown_file_path'].stat().st_mtime
            > fileInfo['source'].stat().st_mtime):
            return True

        if failure_log is not None:
            print(f"{fileInfo['CiteKey']}: dest file not up to date")
            add_fail(failure_log, fileInfo, dict(fail_type='Dest not updated'))

    return False

def add_fail(fail_list, fileInfo, failInfo):
    fail_list.append(failInfo | fileInfo)

def is_markdown_file_big_enough(fileInfo, failure_log=None):
    """Return True if fileInfo['markdown_file_path'] is as big as most quality-converted markdown files."""

    mdFNm = fileInfo['markdown_file_path']
    nbytes_mdf = mdFNm.stat().st_size if mdFNm.exists() else 0
    if nbytes_mdf >= rfw.min_bytes_for_high_quality_html2md:
        return True

    if failure_log is not None:
        print(f"{fileInfo['CiteKey']}: html2md converted file: {nbytes_mdf=} < {rfw.min_bytes_for_high_quality_html2md}")
        add_fail(failure_log, fileInfo, dict(fail_type='Markdown file nonexistent or too short', nbyte=nbytes_mdf))

    return False

#### For each parent item, make a RAG-ready markdown file from the desired attachment.

In [6]:
parentCitekeys = attachments.parentCitekey.unique()
attachments2 = attachments.set_index(['parentCitekey', 'contentType'])

failure_log, cached_files = [], []
for pckey in parentCitekeys:
    attachmentsThis = attachments2.loc[pckey]
    markdown_file_path = rfw.attachments_as_md_cachedir / f'{pckey}.md'
    thisRow = attachmentsThis.iloc[0] # to make it a series
    fileInfo = dict(Title=thisRow.parentTitle, Author=thisRow.parentFirstCreator,
                    Date=thisRow.parentDate, ZoteroKey=thisRow.parentZotkey,
                    Collections=thisRow.parentCollections, 
                    CiteKey=pckey, markdown_file_path=markdown_file_path)

    # Videos
    if (len(attachmentsThis) == 1) and (attachmentsThis.index[0] == 'youtube_video'):
        #attachmentsThishis.reset_index() # index is 'contentType: CHANGE THIS:
        videoURL = attachmentsThis.iloc[0].parentURL
        fileInfo['source'] = videoURL
        print(f"{videoURL} --> {markdown_file_path.name}")
        try:
            rfw.youtube2md(videoURL, markdown_file_path)
            if is_markdown_file_big_enough(fileInfo, failure_log):
                cached_files.append(fileInfo)
        except Exception as e:
            print(f"Conversion exception on {fileInfo['source']} ({e})")
            break
 
        continue # done w/ this pkey

    # PDF or html
    source_file=None
    for source_type in convert_source_to_markdown_func.keys():
        try:
            source_file = attachmentsThis.loc[source_type,'file_fullpath']
            if isinstance(source_file, pd.Series):
                source_file = source_file.iloc[0]
            break # found the highest priorty attachment type
        except:
            continue # next attachment type

    if source_file is None or not source_file.exists() or not isinstance(source_file, pl.WindowsPath):
        print(f'Skipping {pckey=}: bad {str(source_file)=}')
        add_fail(failure_log, fileInfo, dict(fail_type=f'bad {str(source_file)=}'))
        continue # skip whole pkey

    fileInfo = fileInfo | dict(source=source_file, markdown_file_path_file=markdown_file_path)
    if is_markdown_file_big_and_current(fileInfo):
        print(f"Skipping {source_file.name}: Mardkown output file is OK.")
        cached_files.append(fileInfo)
        continue # next pkey

    print(f"{source_file.name} --> {markdown_file_path.name}")

    try:
        convert_source_to_markdown_func[source_type](source_file, markdown_file_path)
        if not markdown_file_path.exists():
            add_fail(failure_log, fileInfo, dict(fail_type=f'missing converted markdown file: {str(markdown_file_path)=}'))
            continue # no file at all not good. Don't merge nothing

        # Have used cautious html2md function, so merge even if small
        cached_files.append(fileInfo) 
    except Exception as e:
        print(f"Conversion exception on {source_file.name} ({e})")
        add_fail(failure_log, fileInfo, dict(fail_type=str(e)))

Mark25unhappyEcon7charts.pdf --> Mark25unhappyEcon7charts.md
Emanuele15DatasetElectoralVolatilitydownload.pdf --> Emanuele15DatasetElectoralVolatilitydownload.md
Lowrey24RiseUnionRighta.html --> Lowrey24RiseUnionRight.md
Klein24itsCorruption.pdf --> Klein24itsCorruption.md
Klein25bidenWhatWentWrong.pdf --> Klein25bidenWhatWentWrong.md
https://www.youtube.com/watch?v=cvWT9SOpwEQ --> Vance25fightBannonMusk.md
Balz25bidenDecRunLegacy.html --> Balz25bidenDecRunLegacy.md
Dionne24hiddenVictoryProgrssiv.html --> Dionne24hiddenVictoryProgrssiv.md
Kolbert25oneEmotionEthics.html --> Kolbert25oneEmotionEthics.md
Montgomery25trumpVotersWant.html --> Montgomery25trumpVotersWant.md
MorningConsult25rigntShiftGenZ.html --> MorningConsult25rigntShiftGenZ.md
Adams92tallerCandidateWins.html --> Adams92tallerCandidateWins.md
Healy24trumpWin61FocusGrps.pdf --> Healy24trumpWin61FocusGrps.md
Harvey25misinfoRenewables.html --> Harvey25misinfoRenewables.md
Igielnik25rememberAboutBiden.html --> Igielnik25rememb

In [7]:
print("previous run: 0 failures (before that it was 62)")

if (nFails := len(failure_log)) < 1:
    print('Completed with no source_file failures')
else:
    print(f'There were {nFails} failures of {len(parentCitekeys)}')
    failure_log = pd.DataFrame(failure_log).set_index('CiteKey')
    display(failure_log.head(3))

previous run: 0 failures (before that it was 62)
Completed with no source_file failures


### Group RAG markdown files, and merge each into a single markdown file

In [8]:
def normalize_weird_line_endings(content):
    """Remove unusual line terminators and normalize to regular newlines."""
    content = content.replace('\u2028', '\n')  # Replace LS
    content = content.replace('\u2029', '\n')  # Replace PS
    content = content.replace('\r\n', '\n').replace('\r', '\n')
    return content

# TODO: ensure that the boundary header is the title, and all the rest of the headers are below that.  
# Hard b/c sometimes md file already has a level 1 header that is the title...
# e.g. Hawkins18hiddenTribes has two copies of its headline in addition to the one I generated.
# this is in the merged result: "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\Politics\Political Causality\Merged RAG Political Sources\merged_RAG_0.md"
def read_and_merge_markdown_files(file_info_list, title):
    """Merge markdown files with their metadata."""

    merged_content_this = ['',f'**{title}**', f'{dt.datetime.now()}', '']

    for file_info in file_info_list:
        # Add metadata at the top of each file's content
        # TODO: would it be better to make the top header the citekey?  This is how I look stuff up in zotero and obsidian
        merged_content_this.append(f"# {file_info['Title']}")
        merged_content_this.append(f"**Author:** {file_info['Author']}, ")
        merged_content_this.append(f"**Date:** {file_info['Date']}")
        merged_content_this.append(f"**Collections:** {", ".join(file_info['Collections'])}")

        citekey, zoterokey = file_info['CiteKey'], file_info['ZoteroKey']
        merged_content_this.append(f"**Zotero Key:** {zoterokey}")
        merged_content_this.append(f"**Cite Key:** {citekey}")

        zotero_link = f"[Zotero Item](zotero://select/library/items/{zoterokey})"
        cite_key_link = f"[Lit Note]({citekey})"
        merged_content_this.append(f"{zotero_link} | {cite_key_link}\n")

        # Read and clean content of the source file
        source_path = Path(file_info['markdown_file_path'])
        with source_path.open('r', encoding='utf-8') as source_file:
            content = source_file.read()

            # fix word wrapping here, without touching metainfo, where this messes up some links
            content = mdformat.text(content, options={"wrap": 80})
            # github standard: even if pop install mdformat-gfm, doesn't work
            # content = mdformat.text(content, extensions=["gfm"])

            content = normalize_weird_line_endings(content)
            # So article title metadata header (above) is always the highest level
            # CHECK this in rag 0 Aldrich18manyFacesStrategicVote
            adjusted_content = rfw.heirarch_shift_markdown_headers(content, 2)
            merged_content_this.append(adjusted_content)

        merged_content_this.append("\n")  # Add a blank line between files

    final_content = '\n'.join(merged_content_this)
    return normalize_weird_line_endings(final_content)

def write_merged_file(merged_content, output_path_md, output_path_pdf=None):
    """Write merged content to file with standard line endings."""
    # Get the string content from merged_content if it's a Path object
    if isinstance(merged_content, Path):
        with merged_content.open('r', encoding='utf-8') as f:
            content = f.read()
    else:
        content = str(merged_content)
    
    content = normalize_weird_line_endings(content)
    
    output_path_md = Path(output_path_md) # 
    with output_path_md.open('w', encoding='utf-8', newline='\n') as output_file:
        output_file.write(content)

    if output_path_pdf is not None:
        output_path_pdf = pl.Path(output_path_pdf)
        rfw.md2pdf_markdown_reportlab(content, output_path_pdf)

##### Group and merge files by word count

In [9]:
# A LITTLE REDUNDANT WITH rfw.count_words_in_markdown(markdown_contend)

def count_words_in_markdown_file(file):
    """
    Counts the words in a single Markdown file.
    
    Args:
        file (str): The path to the Markdown file.
    
    Returns:
        int: The word count of the file.
    
    Raises:
        FileNotFoundError: If the file does not exist.
        Exception: For any other errors while reading or processing the file.
    """
    try:
        with open(file, 'r', encoding='utf-8') as f:
            text = f.read()
            # Remove Markdown syntax
            clean_text = re.sub(r'(\[.*?\]\(.*?\)|[#*`>~\-]|!\[.*?\]\(.*?\))', '', text)
            words = clean_text.split()
            return len(words)
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {file}")
    except Exception as e:
        raise Exception(f"An error occurred while processing {file}: {e}")

In [10]:
all_md_files = pd.DataFrame(cached_files).set_index('CiteKey')
all_md_files['nWords'] = all_md_files.markdown_file_path.apply(lambda fNm: count_words_in_markdown_file(fNm))

if any((isTooBig := all_md_files.nWords > maxNwordsMergedGroup)):
    print(f'Excluding {isTooBig.sum()} of {len(isTooBig)} oversized files:')
    display(all_md_files[isTooBig][['Title','Author', 'Date', 'nWords']])
    all_md_files = all_md_files[~isTooBig]
    print('TODO: split the excluded files.  They seem useful.')

In [11]:
fileGroups = rfw.bin_items_FFD(all_md_files['nWords'], maxNwordsMergedGroup)
print(f'Writing {len(fileGroups)} files to: {str(rfw.merged_RAG_source_dir)}\n')

merged_markdown_output_pdf_file = None
groupStats = []
for groupIx, (ilocs, nWords) in fileGroups.iterrows():
    #ic(groupIx, ilocs, nWords)
    group_files = all_md_files.iloc[ilocs].reset_index().to_dict('records')

    merged_markdown_output_md_file = rfw.merged_RAG_source_dir / f'merged_RAG_{groupIx}.md'
    if save_merged_pdfs:
        merged_markdown_output_pdf_file = rfw.merged_RAG_source_dir / f'merged_RAG_{groupIx}.md'

    groupNm = f'Group {groupIx}'
    print(f'{groupNm} --> {merged_markdown_output_md_file.name}')

    merged_content = read_and_merge_markdown_files(group_files, groupNm)
    write_merged_file(merged_content, merged_markdown_output_md_file, merged_markdown_output_pdf_file)

    groupStatsThis = dict(nFiles=len(group_files), 
                         nBytesGrpMdFile=merged_markdown_output_md_file.stat().st_size,
                         nWords=rfw.count_words_in_markdown(merged_content))
    if save_merged_pdfs:
        groupStatsThis['nBytesGrpPdfFile'] = merged_markdown_output_pdf_file.stat().st_size

    groupStats.append(pd.Series(groupStatsThis,name=groupNm))

groupStats = pd.DataFrame(groupStats)
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
    display(groupStats) # printed big enough to show all columns, not necess. all rows

pd.DataFrame(groupStats.sum())

Writing 7 files to: C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\Politics\Political Causality\Merged RAG Political Sources

Group 0 --> merged_RAG_0.md
Group 1 --> merged_RAG_1.md
Group 2 --> merged_RAG_2.md
Group 3 --> merged_RAG_3.md
Group 4 --> merged_RAG_4.md
Group 5 --> merged_RAG_5.md
Group 6 --> merged_RAG_6.md


,nFiles,nBytesGrpMdFile,nWords
Group 0,8,3239597,456285
Group 1,24,6095517,459635
Group 2,34,4210509,490587
Group 3,43,3788506,481018
Group 4,50,4130538,465641
Group 5,100,13862808,471439
Group 6,267,11121582,349949


,0
nFiles,526
nBytesGrpMdFile,46449057
nWords,3174554


##### Group and merge files by zotero category

In [12]:
# # For each collection group, merge its markdown files into a single file
# for cgroup, gcollections in collection_groups_to_get.items():
#     merged_markdown_output_md_file = rfw.merged_RAG_source_dir / f'{cgroup}_merged_RAG.md'
#     if save_merged_pdfs:
#         merged_markdown_output_pdf_file = rfw.merged_RAG_source_dir / f'{cgroup}_merged_RAG.pdf'

#     print(f'{cgroup} --> {merged_markdown_output_md_file.name}')

#     group_files = [file for file in cached_files if set(gcollections).intersection(file['Collections'])]

#     merged_content = read_and_merge_markdown_files(group_files, cgroup)
#     write_merged_file(merged_content, merged_markdown_output_md_file, merged_markdown_output_pdf_file)

#     groupStatThis = dict(nFiles=len(group_files), 
#                          nBytesGrpMdFile=merged_markdown_output_md_file.stat().st_size,
#                          nWords=rfw.count_words_in_markdown(merged_content),
#                          collections=gcollections)
#     if save_merged_pdfs:
#         groupStatThis['nBytesGrpPdfFile'] = merged_markdown_output_pdf_file.stat().st_size
#     groupStats.append(pd.Series(groupStatThis,name=cgroup))

# groupStats = pd.DataFrame(groupStats)
# with pd.option_context('display.max_columns', None, 'display.max_colwidth', None):
#     display(groupStats) # printed big enough to show all columns, not necess. all rows

##### Merge all files into a single file

In [13]:
# merged_markdown_output_file = rfw.merged_RAG_source_dir / 'merged_RAG.md'
# print(f"Writing merged markdown file to: {merged_markdown_output_file}")
# merged_content = read_and_merge_markdown_files(cached_files)
# write_merged_file(merged_content, merged_markdown_output_file)
# print("Done.")

### Messing aronund with printing clickable filepath links

It's not useful yet

In [14]:
# from pathlib import Path
# import os

# def create_clickable_path(file_path, link_name=None):
#     """
#     Create a clickable link for a file path.
    
#     Args:
#     file_path (str or Path): The path to the file.
#     link_name (str, optional): The text to display for the link. If None, uses the file path.
    
#     Returns:
#     str: HTML string with a clickable link.
#     """
#     # Convert to Path object if it's a string
#     if isinstance(file_path, str):
#         file_path = Path(file_path)
    
#     # Ensure it's an absolute path
#     file_path = file_path.absolute()
    
#     # Convert to URL format
#     if os.name == 'nt':  # Windows
#         url_path = f"file:///{str(file_path).replace('\\', '/').replace(' ', '%20')}"
#     else:  # Unix-like systems (Linux, macOS)
#         url_path = f"file://{str(file_path).replace(' ', '%20')}"
    
#     # Use file path as link name if not provided
#     if link_name is None:
#         link_name = str(file_path)
    
#     # Create HTML link
#     html_link = f'<a href="{url_path}">{link_name}</a>'
    
#     return html_link

# # To use in a Jupyter notebook, you can then do:
# from IPython.display import display, HTML

# def display_clickable_path(file_path, link_name=None):
#     """
#     Display a clickable link for a file path in a Jupyter notebook.
    
#     Args:
#     file_path (str or Path): The path to the file.
#     link_name (str, optional): The text to display for the link. If None, uses the file path.
#     """
#     html_link = create_clickable_path(file_path, link_name)
#     display(HTML(html_link))


In [5]:
# opens folders inside of vscode.
#
# from pathlib import Path
# import os
# from IPython.display import display, HTML

# def create_clickable_path(file_path, link_name=None):
#     """
#     Create a clickable link for a file or folder path.
#     """
#     if isinstance(file_path, str):
#         file_path = Path(file_path)
    
#     file_path = file_path.absolute()
    
#     if link_name is None:
#         link_name = str(file_path)
        
#     # Different handling for directories vs files
#     if file_path.is_dir():
#         if os.name == 'nt':  # Windows
#             url = f"vscode://file/{str(file_path).replace('\\', '/').replace(' ', '%20')}"
#         else:  # Unix-like systems
#             url = f"vscode://file{str(file_path).replace(' ', '%20')}"
#     else:
#         if os.name == 'nt':  # Windows
#             url = f"file:///{str(file_path).replace('\\', '/').replace(' ', '%20')}"
#         else:  # Unix-like systems
#             url = f"file://{str(file_path).replace(' ', '%20')}"
    
#     html_link = f'<a href="{url}" target="_blank">{link_name}</a>'
#     return html_link

# def display_clickable_path(file_path, link_name=None):
#     """
#     Display a clickable link for a fible or folder path in a Jupyter notebook.
#     """
#     html_link = create_clickable_path(file_path, link_name)
#     display(HTML(html_link))


In [7]:
# Example usage:
file_path = r"C:\Users\YourName\Documents\example.txt"  # or "/home/user/documents/example.txt" for Unix-like systems
file_path = rfw.merged_RAG_source_dir
display_clickable_path(file_path)

# Or with a custom link name:
display_clickable_path(file_path, "Click here to open the file")


In [12]:

import os
import platform
from pathlib import Path

def get_open_folder_command(path):
    abs_path = str(Path(path).resolve())
    if not os.path.exists(abs_path):
        return f"Path does not exist: {path}"
    
    if not os.path.isdir(abs_path):
        abs_path = os.path.dirname(abs_path)
    
    system = platform.system().lower()
    if system == 'windows':
        return f'explorer "{abs_path}"'
    elif system == 'darwin':  # macOS
        return f'open "{abs_path}"'
    elif system == 'linux':
        return f'xdg-open "{abs_path}"'
    else:
        return f"Unsupported operating system: {system}"

# Example usage
print(get_open_folder_command(file_path))


explorer "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\Politics\Political Causality\Merged RAG Political Sources"


In [ ]:

# Example usage
print(make_clickable_path(file_path))
